In [3]:
!pip install shapely pyproj

In [1]:
"""
Wellington Street — matching counter 32493 to the correct street segment

Question: counter 32493 sits on Wellington Street, which has 3 treatment
segments (5881, 5882, 5883) with different intervention dates:
    - 5881 / 5882: Painted -> Protected bike lane in 2015
    - 5883:        Painted -> Protected bike lane in 2019

Which segment is the counter actually on? We answer this by measuring the
GPS distance from the counter's coordinates to each segment's road geometry
and picking whichever segment is closest.

Inputs:
    - Bike Site Number Listing (CSV) -> counter GPS_LAT / GPS_LONG
    - streets_spatial.xlsx           -> segment geometry (wkt_geom column)
"""

import pandas as pd
from shapely import wkt
from shapely.geometry import Point
from pyproj import Transformer

# ---------------------------------------------------------------
# 1. Load segment geometry for the 3 Wellington Street segments
# ---------------------------------------------------------------
streets = pd.read_excel("streets_spatial.xlsx")
wellington_segments = streets[streets["street_segment_id"].isin([5881, 5882, 5883])].copy()
wellington_segments["geom"] = wellington_segments["wkt_geom"].apply(wkt.loads)

# ---------------------------------------------------------------
# 2. Counter 32493's two GPS points, from the Bike Site Number Listing
#    (one row per direction: North-bound, South-bound)
# ---------------------------------------------------------------
counter_points = {
    "32493_North_bound": (-37.807171, 144.985903),
    "32493_South_bound": (-37.807664, 144.985967),
}

# ---------------------------------------------------------------
# 3. Reproject counter GPS points (EPSG:7844, GDA2020 lat/lon) into
#    the same CRS as the segment geometry (EPSG:7899, GDA2020 / Vicgrid)
#    so distances come out in metres.
# ---------------------------------------------------------------
transformer = Transformer.from_crs("EPSG:7844", "EPSG:7899", always_xy=True)

# ---------------------------------------------------------------
# 4. Compute distance from each counter point to each segment's line
# ---------------------------------------------------------------
results = []
for point_name, (lat, lon) in counter_points.items():
    x, y = transformer.transform(lon, lat)
    counter_point = Point(x, y)

    for _, seg in wellington_segments.iterrows():
        dist_m = counter_point.distance(seg["geom"])
        results.append(
            {
                "counter_point": point_name,
                "segment_id": seg["street_segment_id"],
                "distance_m": round(dist_m, 2),
            }
        )

results_df = pd.DataFrame(results).pivot(
    index="counter_point", columns="segment_id", values="distance_m"
)

print("Distance (metres) from counter 32493 to each Wellington St segment:\n")
print(results_df.to_string())
print()

closest = results_df.idxmin(axis=1)
print("Closest segment per counter point:")
print(closest.to_string())

# ---------------------------------------------------------------
# Result:
#   Both directions of counter 32493 are ~6-7m from segment 5881,
#   vs 67-120m from 5882 and 309-362m from 5883.
#   -> Counter 32493 is matched to segment 5881.
#
# Segment 5881's intervention history (from sites_db.csv):
#   2015-02-05: Painted bike lane
#   2015-09-13: Protected bike lane   <- confirmed intervention date
#   2019-08-31: Post-protected (no change) -- unrelated disruption, not
#               a new intervention. The 2019 conversion date belongs to
#               segment 5883, not 5881.
# ---------------------------------------------------------------


Distance (metres) from counter 32493 to each Wellington St segment:

segment_id         5881    5882    5883
counter_point                          
32493_North_bound  6.87   67.34  309.08
32493_South_bound  6.01  120.62  362.53

Closest segment per counter point:
counter_point
32493_North_bound    5881
32493_South_bound    5881
